# Week 7 — Merging DataFrames: Joining Olist Tables Together
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Understand inner, left, right, and outer joins
- Use `pd.merge()` correctly
- Detect and handle unmatched rows after a join
- Merge on different key column names

## Why this matters

In the Olist dataset, no single table tells the whole story. The `orders` table knows
*when* an order happened, but not *which state* the customer lives in — that lives in the
`customers` table. The `order_items` table knows *what was sold and for how much*, but the
product's **category** sits in a separate `products` table, and its English name is in yet
another `translation` table.

To answer a business question like *"Which Brazilian state generates the most revenue, and
in which product categories?"* we have to stitch these tables back together on their shared
keys. That stitching is called **merging** (a database *join*), and it is the single most
important skill for turning scattered tables into one analysis-ready DataFrame.

In [2]:
"""
Standard Olist Data Loading for Google Colab

Use this code snippet in every notebook from Week 3 onwards.
"""

import pandas as pd
from google.colab import drive
import os

# Mount Google Drive
#drive.mount('/content/drive')

# Define paths
olist_path = '/content/drive/MyDrive/olist-data'

# Load all datasets
orders = pd.read_csv(os.path.join(olist_path, '/content/olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(olist_path, '/content/olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(olist_path, '/content/olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(olist_path, '/content/olist_products_dataset.csv'))
reviews = pd.read_csv(os.path.join(olist_path, '/content/olist_order_reviews_dataset.csv'))
payments = pd.read_csv(os.path.join(olist_path, '/content/olist_order_payments_dataset.csv'))
sellers = pd.read_csv(os.path.join(olist_path, '/content/olist_sellers_dataset.csv'))
product_translations = pd.read_csv(os.path.join(olist_path, '/content/product_category_name_translation.csv'))

# Verify the two tables we start with today
print(f"Orders:    {orders.shape}")      # expected: (99441, 8)
print(f"Customers: {customers.shape}")    # expected: (99441, 5)
print(f"Items:     {order_items.shape}")  # expected: (112650, 7)
print(f"Products:  {products.shape}")     # expected: (32951, 9)
print(f"Product_translations: {product_translations.shape}")  # expected: (71,2)
print(f"Payments: {payments.shape}")       # expected: (103886, 5)
print(f"Reviews: {reviews.shape}")         # expected: (99224, 7)
print(f"Sellers: {sellers.shape}")         # expected: (3095, 4)

Orders:    (99441, 8)
Customers: (99441, 5)
Items:     (112650, 7)
Products:  (32951, 9)
Product_translations: (71, 2)
Payments: (103886, 5)
Reviews: (99224, 7)
Sellers: (3095, 4)


## 1. The four join types

A **join** answers one question: *when I line up two tables by a shared key, which rows do I keep?*
Think of two guest lists for a party — one from the bride, one from the groom. An **inner join**
is the people who appear on *both* lists (mutual friends). A **left join** keeps *everyone the
bride invited*, and fills in blanks for those the groom didn't know. A **right join** is the
mirror image, and an **outer join** is *everyone from either list*, blanks and all.

In pandas the four types are: `how='inner'` (only matching keys — the default), `how='left'`
(all left rows + matches), `how='right'` (all right rows + matches), and `how='outer'` (all rows
from both). Wherever a row has no match, pandas fills the missing columns with `NaN`. The tiny
illustrative tables below make the difference concrete before we scale up to 100,000 rows.

In [3]:
# ── Illustrative example (small invented tables, NOT Olist) ──────
left = pd.DataFrame({'key': ['a', 'b', 'c'], 'left_val': [1, 2, 3]})
right = pd.DataFrame({'key': ['b', 'c', 'd'], 'right_val': [20, 30, 40]})

print("INNER (keys in both: b, c):")
print(left.merge(right, on='key', how='inner'))  # expected: 2 rows (b, c)

print("\nLEFT (all left keys a, b, c; a gets NaN):")
print(left.merge(right, on='key', how='left'))    # expected: 3 rows, right_val NaN for a

print("\nOUTER (all keys a, b, c, d):")
print(left.merge(right, on='key', how='outer'))   # expected: 4 rows, NaN where unmatched

INNER (keys in both: b, c):
  key  left_val  right_val
0   b         2         20
1   c         3         30

LEFT (all left keys a, b, c; a gets NaN):
  key  left_val  right_val
0   a         1        NaN
1   b         2       20.0
2   c         3       30.0

OUTER (all keys a, b, c, d):
  key  left_val  right_val
0   a       1.0        NaN
1   b       2.0       20.0
2   c       3.0       30.0
3   d       NaN       40.0


## 2. Merging orders with customers (inner join)

Now the real thing. Both `orders` and `customers` are keyed on `customer_id`, and every order
in Olist has exactly one matching customer — so an **inner join** (the default) loses nothing.
Merging is like clipping two index cards together along their shared ID: the result carries all
the columns of both cards on a single row.

Because both tables have 99,441 rows and every key matches, the merged table also has 99,441
rows. Its column count is `8 + 5 − 1 = 12` (the shared `customer_id` key is not duplicated).
Once merged, we can finally group orders by the customer's *state* — something neither table
could do alone.

In [4]:
# Inner merge (how='inner' is the default, so we can omit it)
oc = orders.merge(customers, on='customer_id')

print(orders.shape)     # expected: (99441, 8)
print(customers.shape)  # expected: (99441, 5)
print(oc.shape)         # expected: (99441, 12)
# Same row count: every customer_id in orders exists in customers

# Which columns did the merge add?
new_cols = [c for c in oc.columns if c not in orders.columns]
print(new_cols)
# expected: ['customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

# Now we can group by state
state_orders = oc.groupby('customer_state')['order_id'].count().sort_values(ascending=False)
print(state_orders.head())
# expected: SP 41746, RJ 12852, MG 11635, RS 5466, PR 5045

(99441, 8)
(99441, 5)
(99441, 12)
['customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: order_id, dtype: int64


## 3. Chaining left joins: items → products → translation

To analyse **revenue by product category** we need three tables. `order_items` has the `price`,
`products` has the `product_category_name`, and `product_translations` maps that Portuguese name
to English. We chain the merges.

Here we deliberately use `how='left'` instead of an inner join. Some items point to products whose
category is missing in the data, and we do **not** want to silently drop those sales — that would
understate revenue. A left join keeps every one of the 112,650 items; unmatched ones simply get
`NaN` for the category. Watch: after joining to `products`, exactly **1,603** items have no
category name. Keeping them (rather than losing them to an inner join) is the whole point.

In [5]:
# Step 1: items + products (left join — keep ALL items even if category unknown)
items_products = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)
print(items_products.shape)  # expected: (112650, 8)
print(items_products['product_category_name'].isnull().sum())  # expected: 1603

# Step 2: + translation to get the English category name
items_cat = items_products.merge(
    product_translations,
    on='product_category_name',
    how='left'
)
print(items_cat.shape)  # expected: (112650, 9)

# Revenue by (English) category
cat_revenue = items_cat.groupby('product_category_name_english')['price'].sum()
print(cat_revenue.nlargest(5).round(2))
# expected: health_beauty 1258681.34, watches_gifts 1205005.68,
#           bed_bath_table 1036988.68, sports_leisure 988048.97,
#           computers_accessories 911954.32

(112650, 8)
1603
(112650, 9)
product_category_name_english
health_beauty            1258681.34
watches_gifts            1205005.68
bed_bath_table           1036988.68
sports_leisure            988048.97
computers_accessories     911954.32
Name: price, dtype: float64


## Using AI assistance (DeepSeek)

From Week 4 you may use **DeepSeek** to help write and debug merge code. Follow the protocol —
it keeps you in control of the code, not the other way around:

1. **Understand first** — describe in plain English what you want *before* prompting.
2. **Prompt with context** — give the DataFrame names, the key column(s), and the join type.
3. **Validate the output** — check the result against the verified number from this notebook.
4. **Never copy blindly** — you must be able to explain every line DeepSeek gives you.

The rule that matters most: **if DeepSeek's answer doesn't match the curriculum's verified value,
the code is wrong — not the curriculum.** The cell below shows the Week 7 sample prompt and how to
validate what comes back.

In [6]:
# ── Sample DeepSeek prompt (Week 7) ─────────────────────────────
# "I have two pandas DataFrames, `orders` and `customers`, that share a
#  column called customer_id. How do I merge them and show the shape
#  and the first 3 rows?"
#
# DeepSeek should return something equivalent to:
oc_check = orders.merge(customers, on='customer_id')
print(oc_check.shape)  # expected: (99441, 12)

# VALIDATE: does the shape match our verified value (99441, 12)?
assert oc_check.shape == (99441, 12), "Shapes differ — revise the prompt, trust the curriculum"
print("Validated: DeepSeek output matches the verified merge shape")  # expected: Validated: ...

(99441, 12)
Validated: DeepSeek output matches the verified merge shape


## 4. Going deeper: different key names & spotting unmatched rows

Two situations trip up real analysts. **First**, the join key often has *different names* in the
two tables (e.g. `customer_id` in one, `cust_id` in another). You can't use `on=` then — you use
`left_on=` and `right_on=`. Note that pandas keeps *both* key columns in the result, so the column
count goes up by one versus a same-name merge.

**Second**, after a left join you should always *count* the unmatched rows rather than assume the
join was clean. The `indicator=True` flag adds a `_merge` column labelling each row `both`,
`left_only`, or `right_only` — the fastest way to audit a join. Here it confirms exactly 1,603
items had no product match (`112650 − 1603 = 111047` matched).

In [7]:
# (a) Merge on DIFFERENT key column names with left_on / right_on
customers_renamed = customers.rename(columns={'customer_id': 'cust_id'})
oc_diff = orders.merge(customers_renamed, left_on='customer_id', right_on='cust_id')
print(oc_diff.shape)  # expected: (99441, 13)  <- one extra col: both keys kept

# (b) Audit unmatched rows with indicator=True
audit = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id', how='left', indicator=True
)
print(audit['_merge'].value_counts())
# expected: both 111047, left_only 1603, right_only 0

(99441, 13)
_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


## 5. Common mistakes

The number-one merge bug is **silently losing rows** by forgetting `how='left'` and getting an
inner join instead. The number-two bug is confusing **duplicate column suffixes** (`_x`, `_y`)
that pandas adds when both tables share a non-key column. Study the annotated cell below — the
errors are shown as comments so nothing crashes.

In [8]:
# ── COMMON MISTAKE 1: inner join silently drops unmatched rows ───
# WRONG — default inner join loses the 1,603 category-less items:
# bad = order_items.merge(products[['product_id','product_category_name']], on='product_id')
# print(bad.shape)   # (111047, 8)  <- 1,603 sales vanished!

# CORRECT — use how='left' to keep every item:
good = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)
print(good.shape)  # expected: (112650, 8)

# ── COMMON MISTAKE 2: surprise _x / _y suffix columns ────────────
# When both tables share a NON-key column, pandas renames them _x and _y.
# Fix it up front by selecting only the columns you need (as above),
# or pass suffixes=('_item', '_prod') to label them clearly.
demo = order_items.merge(products, on='product_id', how='left', suffixes=('_item', '_prod'))
print([c for c in demo.columns if c.endswith(('_item', '_prod'))])  # expected: [] (no shared non-key cols here)

(112650, 8)
[]


## 6. Mini-challenge  ⏱ ~5 minutes

Using the `orders` and `customers` tables, merge them on `customer_id` and find out **how many
orders came from Rio de Janeiro state (`RJ`)**.

**Expected output:** `12852`

Fill in the two blanks below.

In [10]:
# Given (do not change these):
#   orders     — the orders table
#   customers  — the customers table

# 1. Merge orders and customers on customer_id
mini = orders.merge(customers, on='customer_id')  # your merge here


# 2. Count how many rows have customer_state == 'RJ'
rj_orders = mini[mini['customer_state'] == 'RJ'].shape[0]  # your code here
print(rj_orders)  # expected: 12852

12852


## 7. Group exercise (25 min)

**Build the full merged table, step by step**, then use it to answer two business questions.
Work together — one person drives, the rest verify each shape before moving on.

Merge in this order:
1. `orders` → merge `customers` (on `customer_id`)
2. → merge `order_items` (on `order_id`)
3. → merge `products[['product_id', 'product_category_name']]` (on `product_id`, `how='left'`)
4. → merge `product_translations` (on `product_category_name`, `how='left'`)

**The final shape must be `(112650, 20)`.** Verify it before answering.

Then answer:
- **What is the total revenue from SP state?**  *(Expected: R$5,202,955.05)*
- **What is the top category by revenue in RJ state?**

In [14]:
full = (
    orders
    .merge(customers, on='customer_id')
    .merge(order_items, on='order_id')
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(product_translations, on='product_category_name', how='left')
)
print(full.shape)  # expected: (112650, 20)

(112650, 20)


## Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Inner join (default) | `a.merge(b, on='key')` | `orders.merge(customers, on='customer_id')` |
| Left join (keep all left) | `a.merge(b, on='key', how='left')` | `order_items.merge(products, on='product_id', how='left')` |
| Different key names | `a.merge(b, left_on=..., right_on=...)` | `orders.merge(c, left_on='customer_id', right_on='cust_id')` |
| Audit unmatched rows | `a.merge(b, ..., indicator=True)` | `audit['_merge'].value_counts()` |
| Chain merges | `df.merge(...).merge(...)` | build the `(112650, 20)` full table |

---
**Coming up Thursday**: we chain *all* the tables into one pipeline, meet `pd.concat()` for
**stacking** DataFrames, handle mismatched columns after a concat, and run a full multi-table
analysis (revenue + average review score + payment-type breakdown by state).